In [ ]:
import os
import pandas as pd
import json
import random
from datasets import load_dataset

# 1. DOSSIERS
folders = ["data/raw", "data/processed", "data/splits"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# 2. CHARGEMENT MASAKHANE (Nouvelle méthode sans URL directe)
print("--- Étape 1 : Chargement de Masakhane ---")
masakhane_list = []
try:
    # On utilise load_dataset mais on force le format 'pandas' pour éviter les scripts .py
    ds = load_dataset("masakhane/mafand", "fra-bam", split="train", trust_remote_code=True)
    for entry in ds:
        masakhane_list.append({
            "fra": entry["translation"]["fra"],
            "bam": entry["translation"]["bam"]
        })
    print(f"Succès : {len(masakhane_list)} phrases Masakhane récupérées.")
except Exception as e:
    print(f"Erreur Masakhane : {e}. On continue avec le local.")

# 3. TON LEXIQUE LOCAL (Flexible sur les noms de colonnes)
print("--- Étape 2 : Chargement du lexique local ---")
try:
    local_df = pd.read_csv('mon_lexique.csv')

    # On détecte automatiquement les colonnes (on prend la 1ère et la 2ème)
    col_fra = local_df.columns[0]
    col_diou = local_df.columns[1]

    local_list = [{"fra": row[col_fra], "bam": row[col_diou]} for _, row in local_df.iterrows()]
    print(f"Succès : {len(local_list)} mots locaux ajoutés (Colonnes détectées : {col_fra}, {col_diou}).")
except Exception as e:
    print(f"Erreur locale critique : {e}")
    local_list = []

# 4. FUSION ET SAUVEGARDE
print("--- Étape 3 : Fusion ---")
all_data = masakhane_list + local_list
chatbot_data = []

if not all_data:
    print("ERREUR : Aucune donnée trouvée. Vérifie l'upload de mon_lexique.csv !")
else:
    for item in all_data:
        chatbot_data.append({
            "instruction": f"Comment dit-on en Dioula : {item['fra']} ?",
            "response": f"En Dioula, on dit : {item['bam']}."
        })

    random.shuffle(chatbot_data)
    with open("data/processed/final_chatbot_data.jsonl", "w", encoding="utf-8") as f:
        for entry in chatbot_data:
            json.dump(entry, f, ensure_ascii=False)
            f.write("\n")
    print(f"--- TERMINÉ : {len(chatbot_data)} lignes prêtes ! ---")